In [21]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score,precision_score,recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,DataCollatorWithPadding,EarlyStoppingCallback)
from peft import (LoraConfig,get_peft_model,TaskType)

In [22]:
LORA_DATASET_FILE = "../Dane/ALL_DATA_20.04.2026.csv" # ./source/pkd.csv | ../Dane/ALL_DATA_20.04.2026.csv 
MODEL_NAME = "sdadas/polish-roberta-base-v2"
MAX_LEN = 128
label_encoder = LabelEncoder()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir = "./model/", use_fast=True)

In [23]:
data = pd.read_csv(LORA_DATASET_FILE, sep=';')
data = data[['PKD_2007','description_PL']]
data = data.rename(columns={
        'PKD_2007': 'kod_pkd', 
        'description_PL': 'opis'
    })
data['kod_pkd'] = data['kod_pkd'].astype(str).str.strip()
data['opis'] = (
        data['opis']
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
data = data[data['opis'].str.len() > 20]
data = data[['kod_pkd', 'opis']].dropna()
data = data.drop_duplicates()
data["labels"] = label_encoder.fit_transform(data["kod_pkd"])
data.to_csv("labels_encoder.csv",index_label="id")
NUM_LABELS = len(label_encoder.classes_)
print(NUM_LABELS)

262


In [24]:
df_train, df_test = train_test_split(data,test_size=0.1,stratify=data["labels"])

In [25]:
df_train.to_csv('../Dane/teach_set.csv')
df_test.to_csv('../Dane/test_set.csv')

In [26]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df_train["labels"]),
    y=df_train["labels"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)


In [27]:
class WeightedTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss
    

df_train["text"] = df_train["opis"].astype(str)
df_test["text"] = df_test["opis"].astype(str)
train_dataset = Dataset.from_pandas(
    df_train[["text", "labels"]]
)

test_dataset = Dataset.from_pandas(
    df_test[["text", "labels"]]
)

In [28]:
lengths = data["opis"].apply(lambda x: len(tokenizer.tokenize(x)))
print(lengths.describe())

count    79289.000000
mean        26.940862
std         34.066612
min          3.000000
25%         12.000000
50%         17.000000
75%         24.000000
max       1669.000000
Name: opis, dtype: float64


In [29]:
print(lengths.quantile([0.90, 0.95, 0.99, 0.995, 0.999]))

0.900     52.000
0.950     84.000
0.990    165.000
0.995    178.000
0.999    268.424
Name: opis, dtype: float64


In [30]:
counts = data["labels"].value_counts()

print(counts.describe())
print(counts.quantile([0.1,0.25,0.5,0.75,0.9,0.95,0.99]))

count     262.000000
mean      302.629771
std       277.316458
min         2.000000
25%       137.250000
50%       228.500000
75%       404.500000
max      1962.000000
Name: count, dtype: float64
0.10      41.20
0.25     137.25
0.50     228.50
0.75     404.50
0.90     587.20
0.95     694.95
0.99    1365.17
Name: count, dtype: float64


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, cache_dir = "./model/")
# Konfiguracja LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32, # Parametr odpowiedzialny za ilość trenowanych parametrów, im większe r tym ilość parametrów oryginalengo modelu zostaje zwiększona.
    lora_alpha=64, #Parametr odowiedzialny za to jak mocno wyuczone zmiany wpływają na model
    lora_dropout=0.1, # Parametr odpowiedzilany za regularizacje, zmniejsza ryzyko przeuczenia

    # dla Roberta:
    target_modules=["query", "value","key"],
    modules_to_save=["classifier"],
    bias="none"
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2008.33it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: sdadas/polish-roberta-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 2,561,542 || all params: 127,205,900 || trainable%: 2.0137


In [32]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


Map: 100%|██████████| 7929/7929 [00:00<00:00, 18799.24 examples/s]


In [33]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "precision_macro": precision_score(labels,preds,average="macro",zero_division=0),
        "recall_macro": recall_score(labels,preds,average="macro",zero_division=0)
    }

In [34]:
# Parametryzacja treningu 
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate=3e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    num_train_epochs=8,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    logging_steps=50,
    
    save_total_limit=2,
    greater_is_better=True,
    report_to="none",
    warmup_steps=200,

    fp16=torch.cuda.is_available()
)
# Wybór trenera 
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./final_model_GPU")
tokenizer.save_pretrained("./final_model_GPU")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
1,75.379268,4.569898,0.160676,0.047905,0.093609,0.070100,0.076658
2,65.404414,3.888311,0.248329,0.112269,0.178122,0.154778,0.148028
3,58.305781,3.489793,0.288309,0.158294,0.221709,0.200284,0.198475
4,53.943052,3.177877,0.350107,0.221935,0.295300,0.275494,0.256236
5,49.915210,2.970022,0.379997,0.257180,0.331760,0.313300,0.292140


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorWithPadding(tokenizer)
)

# prediction output
pred_output = trainer.predict(test_dataset)

# logits -> klasy
y_pred = np.argmax(pred_output.predictions, axis=-1)

# prawdziwe etykiety
y_true = pred_output.label_ids

print(confusion_matrix(y_true, y_pred))

print(classification_report(y_true, y_pred))

print(y_pred)

In [ ]:
report =classification_report(y_true, y_pred, output_dict=True)
df_report = pd.DataFrame(report).transpose()

df_report.to_csv("classification_report.csv", index=True)

In [ ]:
from sklearn.metrics import top_k_accuracy_score

probs = torch.softmax(
    torch.tensor(pred_output.predictions),
    dim=-1
).numpy()

top3 = top_k_accuracy_score(
    y_true,
    probs,
    k=3,
    labels=np.arange(NUM_LABELS)
)

top5 = top_k_accuracy_score(
    y_true,
    probs,
    k=5,
    labels=np.arange(NUM_LABELS)
)
top10 = top_k_accuracy_score(
    y_true,
    probs,
    k=10,
    labels=np.arange(NUM_LABELS)
)

print(top3, top5, top10)

### 

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding

MODEL_PATH = "./final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH,use_fast=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorWithPadding(tokenizer)
)

text = "Branża: Meble | Produkty: meble na wymiar | Usługi: projektowanie mebli"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=False
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

y_pred = outputs.logits.argmax(dim=-1).cpu().numpy()

print(y_pred)

In [ ]:
data = pd.read_csv('./labels_encoder.csv')
data.loc[data['labels']==y_pred[0]].head(1)